## Cell 0 — Path Configuration

This cell defines the canonical path registry used throughout the notebook.
All subsequent cells read paths from this configuration to ensure portability and reproducibility.

In [ ]:
# Cell 0 — Path Configuration

import os
from dataclasses import dataclass, field
from pathlib import Path

@dataclass
class ExpPaths:
    base:       str = "/home/aggelosger"
    cache_root: str = ""
    outdir:     str = ""
    split_dir:  str = ""
    csv_dirs:   list = field(default_factory=list)

    def __post_init__(self):
        self.cache_root = os.path.join(self.base, "mel_cache_ast16k_256")
        self.outdir     = os.path.join(self.base, "EXP1_AST_ENSEMBLE_SIZE")
        self.split_dir  = os.path.join(self.outdir, "exp1_splits")

        self.csv_dirs = []
        for group in [
            "harmonic_intervals_loose",
            "triads_loose",
            "7th_chords_loose",
        ]:
            d = os.path.join(self.base, group, "csv")
            if os.path.isdir(d):
                self.csv_dirs.append(d)

    def verify(self):
        errors = []
        if not os.path.isdir(self.cache_root):
            errors.append(f"Cache not found: {self.cache_root}")
        if not self.csv_dirs:
            errors.append("No CSV directories found")
        for d in self.csv_dirs:
            csvs = [f for f in os.listdir(d)
                    if f.endswith(".csv") and "-original" not in f]
            if not csvs:
                errors.append(f"No usable CSVs in {d}")
        if errors:
            for e in errors:
                print(f"  \u274c {e}")
            raise RuntimeError("Path verification failed")

        os.makedirs(self.outdir, exist_ok=True)
        os.makedirs(self.split_dir, exist_ok=True)

        all_csvs = []
        for d in self.csv_dirs:
            all_csvs += [os.path.join(d, f) for f in os.listdir(d)
                         if f.endswith(".csv") and "-original" not in f]

        print(f"  \u2705 Base       : {self.base}")
        print(f"  \u2705 Cache      : {self.cache_root}")
        print(f"  \u2705 Output     : {self.outdir}")
        print(f"  \u2705 Splits     : {self.split_dir}")
        print(f"  \u2705 CSV dirs   : {len(self.csv_dirs)}")
        print(f"  \u2705 CSV files  : {len(all_csvs)} (excluding -original)")
        return all_csvs

paths = ExpPaths()
all_csv_paths = paths.verify()
print(f"\n\u2705 Path configuration OK")

  ✅ Base       : /home/aggelosger
  ✅ Cache      : /home/aggelosger/mel_cache_ast16k_256
  ✅ Output     : /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE
  ✅ Splits     : /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE/exp1_splits
  ✅ CSV dirs   : 3
  ✅ CSV files  : 11 (excluding -original)

✅ Path configuration OK


## Cell 0.1 — Environment Check

This cell reports the active Python executable and module search paths.
It is intended to confirm that the runtime environment is correctly configured before execution.

In [ ]:
import sys

print("Python interpreter:")
print(f"  {sys.executable}")
print("\nModule search paths:")
for p in sys.path:
    if p:
        print(f"  {p}")

Python interpreter:
  /usr/bin/python3

Module search paths:
  /usr/lib/python312.zip
  /usr/lib/python3.12
  /usr/lib/python3.12/lib-dynload
  /home/aggelosger/.local/lib/python3.12/site-packages
  /usr/local/lib/python3.12/dist-packages
  /usr/lib/python3/dist-packages


## Cell 3 — Constants, Paths, and CSV Inventory (EXP1: Ensemble Size)

This cell defines shared constants, directory paths, and metadata CSV resources used in EXP1.
These definitions are referenced by downstream preprocessing, training, and evaluation steps.

In [ ]:
# Cell 3 — Constants, paths, CSV inventory (EXP1: ensemble-size)

cell_start("Cell 3")
import os
import pandas as pd
import numpy as np
from dataclasses import dataclass

# ===================== Audio / Feature constants =====================
TARGET_SR     = 16000   # AST-native sample rate
TARGET_FRAMES = 256     # ~2.6 sec at 16 kHz / 10 ms hop (fits 2-sec clips)
N_MELS        = 128
DURATION      = 2.0     # clip length in seconds

# ===================== Reproducibility =====================
SEED = 1337

# ===================== Model =====================
MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
TASK_NAME = "EXP1 ensemble-size classification"
TASK_OBJECTIVE = "Classify whether the audio contains a duo (2), trio (3), or quartet (4) of string instruments."
IGNORE_INDEX = -1
LABEL_SMOOTHING = 0.05

# ===================== Filesystem (from Cell 0) =====================
CACHE_ROOT = paths.cache_root
os.makedirs(paths.outdir, exist_ok=True)

# ===================== Ensemble-size CSV group mapping =====================
# Duos    : harmonic_intervals_loose  (2 instruments)
# Trios   : triads_loose              (3 instruments)
# Quartets: 7th_chords_loose          (4 instruments)

_CSV_MAP = {
    "duos": ("harmonic_intervals_loose", [
        "cello_viola_loose.csv", "cello_violin_loose.csv",
        "viola_violin_loose.csv"]),
    "trios": ("triads_loose", [
        "triads_major_loose.csv", "triads_minor_loose.csv",
        "triads_diminished_loose.csv", "triads_augmented_loose.csv"]),
    "quartets": ("7th_chords_loose", [
        "major_seventh_loose.csv", "minor_seventh_loose.csv",
        "dominant_seventh_loose.csv", "half_diminished_seventh_loose.csv"]),
}

def _build_csv_list(group_key):
    folder, files = _CSV_MAP[group_key]
    return [os.path.join(paths.base, folder, "csv", f) for f in files]

csv_duos     = _build_csv_list("duos")
csv_trios    = _build_csv_list("trios")
csv_quartets = _build_csv_list("quartets")

ALL_CSVS = csv_duos + csv_trios + csv_quartets

# ===================== Fail-fast inventory check =====================
_missing = [p for p in ALL_CSVS if not os.path.isfile(p)]
if _missing:
    print("\u274c Missing CSV files:")
    for p in _missing: print("  -", p)
    raise FileNotFoundError(f"{len(_missing)} CSV(s) missing \u2014 fix paths and re-run this cell.")

print(f"   CSV inventory OK \u2014 {len(ALL_CSVS)} files found.")
print(f"   CACHE_ROOT : {CACHE_ROOT}")
print(f"   OUTDIR     : {paths.outdir}")

cell_end("Cell 3")

[Cell 3]  started  2026-04-11  10:04:01
   CSV inventory OK — 11 files found.
   CACHE_ROOT : /home/aggelosger/mel_cache_ast16k_256
   OUTDIR     : /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE
[Cell 3]  finished in 0.61 sec


## Cell 4 — Load Metadata and Derive Ensemble-Size Labels (EXP1)

This cell loads metadata tables and derives the target label for ensemble-size classification.
The resulting dataframe is used as the primary input for split construction and model training.

In [ ]:
# Cell 4 — Load metadata & derive ensemble-size label (EXP1)

cell_start("Cell 4")

import os, pandas as pd, numpy as np

CANDIDATE_FILENAME_COLS = ["chord_filename", "filename", "wav_filename", "output_wav", "wav_file"]

def resolve_filename_column(df: pd.DataFrame) -> str:
    for c in CANDIDATE_FILENAME_COLS:
        if c in df.columns:
            return c
    raise KeyError(f"Filename column not found in CSV. Tried: {CANDIDATE_FILENAME_COLS}")

def derive_audio_root_from_csv(csv_path: str) -> str:
    csv_dir, csv_file = os.path.split(csv_path)
    base = os.path.splitext(csv_file)[0]
    wav_dir = csv_dir.replace("/csv", "/wav")
    return os.path.join(wav_dir, base) + "/"

def _cache_path(cache_root, csv_path, wav_filename):
    csv_dir = os.path.dirname(csv_path)
    group = os.path.basename(os.path.dirname(csv_dir))
    subset = os.path.splitext(os.path.basename(csv_path))[0]
    stem = os.path.splitext(os.path.basename(str(wav_filename).strip()))[0]
    return os.path.join(cache_root, group, subset, f"{stem}.npy")

def load_ensemble_block(csv_list, ensemble_tag, cache_root):
    # Load a list of CSVs that all belong to the same ensemble-size class.
    frames = []
    for path in csv_list:
        if not os.path.exists(path):
            print(f"   Warning: CSV not found, skipping: {path}")
            continue

        df = pd.read_csv(path)
        fname_col = resolve_filename_column(df)
        audio_root = derive_audio_root_from_csv(path)

        df["ensemble_size"] = ensemble_tag
        df["subset"] = "loose"

        # Build absolute paths
        df["filepath"] = df[fname_col].apply(
            lambda x: str(x).strip() if os.path.isabs(str(x).strip())
            else os.path.join(audio_root, str(x).strip())
        )
        df["cachefile"] = df[fname_col].apply(
            lambda x: _cache_path(CACHE_ROOT, path, x)
        )

        # Keep only rows where WAV exists
        df = df[df["filepath"].apply(os.path.exists)].reset_index(drop=True)

        if len(df) == 0:
            print(f"   Warning: empty block after filtering for {path}")

        frames.append(df)

    return frames


all_dfs = []
all_dfs += load_ensemble_block(csv_duos,     "duo",     CACHE_ROOT)
all_dfs += load_ensemble_block(csv_trios,    "trio",    CACHE_ROOT)
all_dfs += load_ensemble_block(csv_quartets, "quartet", CACHE_ROOT)

if not all_dfs:
    raise FileNotFoundError("No valid data was loaded.")

df_all = pd.concat(all_dfs, ignore_index=True)

print(f"Loaded {len(df_all)} total samples")

print("\nEnsemble-size distribution:")
print(df_all["ensemble_size"].value_counts())

print("\nSubset distribution:")
print(df_all["subset"].value_counts())

print("\nSubset \u00d7 Ensemble size:")
print(pd.crosstab(df_all["subset"], df_all["ensemble_size"]))

cell_end("Cell 4")

[Cell 4]  started  2026-04-11  10:04:11
Loaded 262598 total samples

Ensemble-size distribution:
ensemble_size
quartet    92598
duo        90000
trio       80000
Name: count, dtype: int64

Subset distribution:
subset
loose    262598
Name: count, dtype: int64

Subset × Ensemble size:
ensemble_size    duo  quartet   trio
subset                              
loose          90000    92598  80000
[Cell 4]  finished in 9.51 sec


## Cell 4d — Data Integrity Check

This cell performs consistency checks on the assembled dataframe (schema, missing values, and label validity).
The objective is to verify dataset integrity before split generation and optimization.

In [ ]:
# Cell 4d — Integrity check for df_all
cell_start("Cell 4d")

import numpy as np

# 1) Check ensemble_size labels
expected_types = {"duo", "trio", "quartet"}
observed_types = set(df_all["ensemble_size"].unique())
assert observed_types == expected_types, \
    f"Unexpected ensemble sizes: {observed_types - expected_types}"
print(f"Ensemble sizes OK: {sorted(observed_types)}")

# 2) Check cache files exist
n_missing = (~df_all["cachefile"].apply(os.path.exists)).sum()
print(f"Missing cache files: {n_missing} / {len(df_all)}")
if n_missing > 0:
    print("  \u26a0\ufe0f  Some cache files are missing \u2014 run mel caching first!")

# 3) Spot-check shapes
for i in [0, len(df_all)//2, len(df_all)-1]:
    cf = df_all.iloc[i]["cachefile"]
    if os.path.exists(cf):
        mel = np.load(cf)
        assert mel.shape == (128, 256), f"Bad shape at idx {i}: {mel.shape}"
        assert np.isfinite(mel).all(), f"Non-finite values at idx {i}"

print("Shape & value spot-checks: OK")

cell_end("Cell 4d")

[Cell 4d]  started  2026-04-11  10:04:33
Ensemble sizes OK: ['duo', 'quartet', 'trio']
Missing cache files: 0 / 262598
Shape & value spot-checks: OK
[Cell 4d]  finished in 4.02 sec


## Cell 5 — Stratified Train/Validation/Test Split (70/15/15)

This cell creates a stratified partition of the dataset into training, validation, and test subsets.
Stratification preserves class balance across splits to support reliable evaluation.

In [ ]:
# Cell 5 — Stratified Train / Val / Test Split (70 / 15 / 15)

cell_start("Cell 5")

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

OUT_SPLIT_DIR = Path(paths.split_dir)
OUT_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

_split_file    = OUT_SPLIT_DIR / "exp1_split_indices.npz"
_manifest_file = OUT_SPLIT_DIR / "exp1_manifest.csv"

# ===================== CACHE VALIDATION =====================
use_cache = False

if _split_file.exists() and _manifest_file.exists():
    manifest = pd.read_csv(_manifest_file)
    if len(manifest) == len(df_all):
        use_cache = True
    else:
        print("Cached splits incompatible with current dataset \u2014 recomputing...")
        _split_file.unlink(missing_ok=True)
        _manifest_file.unlink(missing_ok=True)

# ===================== LOAD OR COMPUTE =====================
if use_cache:
    print("Loading existing splits from disk...")
    _npz      = np.load(_split_file)
    train_idx = _npz["train_idx"]
    val_idx   = _npz["val_idx"]
    test_idx  = _npz["test_idx"]

else:
    print("Computing splits...")

    # Stratify by ensemble_size
    strat_key = df_all["ensemble_size"].astype(str)

    idx_all = np.arange(len(df_all))

    train_idx, tmp_idx = train_test_split(
        idx_all, test_size=0.30, random_state=SEED, stratify=strat_key)

    strat_tmp = strat_key.iloc[tmp_idx]
    val_idx, test_idx = train_test_split(
        tmp_idx, test_size=0.50, random_state=SEED, stratify=strat_tmp)

    # Verify disjointness
    S_tr, S_va, S_te = set(train_idx), set(val_idx), set(test_idx)
    assert len(S_tr & S_va) == 0
    assert len(S_tr & S_te) == 0
    assert len(S_va & S_te) == 0
    assert len(S_tr | S_va | S_te) == len(df_all)
    print("Disjointness & coverage: OK")

    # Save
    np.savez_compressed(_split_file,
        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)

    manifest = pd.DataFrame({
        "idx": np.arange(len(df_all)),
        "split": "none",
        "subset": df_all["subset"].values,
        "ensemble_size": df_all["ensemble_size"].values,
    })
    manifest.loc[train_idx, "split"] = "train"
    manifest.loc[val_idx,   "split"] = "val"
    manifest.loc[test_idx,  "split"] = "test"
    manifest.to_csv(_manifest_file, index=False)
    print("Saved splits")

total = len(train_idx) + len(val_idx) + len(test_idx)
print(f"\nSplit sizes \u2014 train: {len(train_idx)} ({len(train_idx)/total:.1%}) "
      f"| val: {len(val_idx)} ({len(val_idx)/total:.1%}) "
      f"| test: {len(test_idx)} ({len(test_idx)/total:.1%})")

# ===================== REPORT =====================
for name, idx in [("TRAIN", train_idx), ("VAL", val_idx), ("TEST", test_idx)]:
    print(f"\n{name} \u2014 ensemble_size counts")
    print(df_all.iloc[idx]["ensemble_size"].value_counts())

cell_end("Cell 5")

[Cell 5]  started  2026-04-11  10:04:48
Computing splits...
Disjointness & coverage: OK
Saved splits

Split sizes — train: 183818 (70.0%) | val: 39390 (15.0%) | test: 39390 (15.0%)

TRAIN — ensemble_size counts
ensemble_size
quartet    64818
duo        63000
trio       56000
Name: count, dtype: int64

VAL — ensemble_size counts
ensemble_size
quartet    13890
duo        13500
trio       12000
Name: count, dtype: int64

TEST — ensemble_size counts
ensemble_size
quartet    13890
duo        13500
trio       12000
Name: count, dtype: int64
[Cell 5]  finished in 3.13 sec


## Cell 11.8 — Target Selection for Ensemble-Size Classification

This cell configures the target field used in EXP1 and aligns the training dataframe accordingly.
It establishes the label interface consumed by the dataset and collator definitions.

In [ ]:
# Cell 11.8 — Focus ensemble-size selection (EXP1)
cell_start("Cell 11.8")

import json, os

if "df_train" not in globals():
    raise RuntimeError("df_train not found. Run Cells 3 \u2192 4 \u2192 5/6b first.")

FOCUS = ["duo", "trio", "quartet"]

observed = sorted(df_train["ensemble_size"].unique())
missing = [x for x in FOCUS if x not in observed]
if missing:
    raise RuntimeError(f"Focus ensemble sizes missing from training data: {missing}")

focus_counts = df_train["ensemble_size"].value_counts()
total = len(df_train)

print(f"FOCUS: {FOCUS}")
for es in FOCUS:
    n = int(focus_counts.get(es, 0))
    print(f"  {es:15s}  {n:>8,}  ({100*n/total:.1f}%)")

os.makedirs(paths.split_dir, exist_ok=True)
with open(os.path.join(paths.split_dir, "focus_labels.json"), "w") as f:
    json.dump({"ensemble_sizes": FOCUS}, f, indent=2)

print("Saved \u2192", os.path.join(paths.split_dir, "focus_labels.json"))

cell_end("Cell 11.8")

[Cell 11.8]  started  2026-04-11  10:05:19
FOCUS: ['duo', 'trio', 'quartet']
  duo                63,000  (34.3%)
  trio               56,000  (30.5%)
  quartet            64,818  (35.3%)
Saved → /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE/exp1_splits/focus_labels.json
[Cell 11.8]  finished in 0.03 sec


## Cell 11.10 — Class Weights for Imbalance Mitigation

This cell computes class weights from the training split to reduce imbalance effects during optimization.
The resulting weights are passed to the loss function in subsequent training cells.

In [ ]:
# Cell 11.10 — Class weights for imbalance (ensemble-size)
cell_start("Cell 11.10")

import numpy as np
import torch
import os, json

assert "FOCUS" in globals()
assert "df_train" in globals()

def _class_weights_from_counts(counts_dict, beta=0.9999):
    labels = list(counts_dict.keys())
    ns = np.array([max(1, counts_dict[k]) for k in labels], dtype=np.float64)
    w = (1.0 - beta) / (1.0 - np.power(beta, ns))
    w = w / w.mean()
    return labels, w.astype(np.float32)

ens_counts = {es: int((df_train["ensemble_size"] == es).sum()) for es in FOCUS}
ens_keys, ens_w = _class_weights_from_counts(ens_counts, beta=0.9999)

assert ens_keys == FOCUS, "Weight order mismatch with FOCUS!"

CLASS_WEIGHTS = torch.tensor(ens_w, dtype=torch.float32)

print("Ensemble-size weights (mean\u22481):", np.round(CLASS_WEIGHTS.numpy(), 3))
print("Counts:", ens_counts)

os.makedirs(paths.split_dir, exist_ok=True)
np.save(os.path.join(paths.split_dir, "class_weights_ensemble.npy"), CLASS_WEIGHTS.numpy())

with open(os.path.join(paths.split_dir, "class_weights_counts.json"), "w") as f:
    json.dump({"ensemble_counts": ens_counts}, f, indent=2)

print("Saved class weights \u2192", paths.split_dir)

cell_end("Cell 11.10")

[Cell 11.10]  started  2026-04-11  10:05:30
Ensemble-size weights (mean≈1): [0.999 1.001 0.999]
Counts: {'duo': 63000, 'trio': 56000, 'quartet': 64818}
Saved class weights → /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE/exp1_splits
[Cell 11.10]  finished in 0.08 sec


## Cell 11.11 — Dataset, Collation, and Split Objects (EXP1)

This cell instantiates dataset objects, collator logic, and split-specific dataloading structures.
These components define the final data interface consumed by the training pipeline.

In [ ]:
# Cell 11.11 — Dataset, collate, split construction (EXP1)
cell_start("Cell 11.11")

import numpy as np
import torch
from torch.utils.data import Dataset

for _n in ("df_train", "df_val", "df_test", "FOCUS"):
    if _n not in globals():
        raise RuntimeError(f"{_n} missing \u2014 run previous cells.")

TARGET_FRAMES = 256

def pad_or_truncate(mel, n_frames):
    t = mel.shape[-1]
    if t >= n_frames:
        return mel[..., :n_frames]
    return torch.nn.functional.pad(mel, (0, n_frames - t))

ens2i = {s: i for i, s in enumerate(FOCUS)}

print("Ensemble-size classes:", FOCUS)
print("ens2i:", ens2i)

class SECDEnsembleDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            mel = torch.tensor(np.load(row["cachefile"]).astype(np.float32))
        except Exception:
            mel = torch.zeros(128, TARGET_FRAMES)

        mel = pad_or_truncate(mel, TARGET_FRAMES)

        label = ens2i.get(row["ensemble_size"], -1)

        return {
            "input_values": mel,
            "labels": torch.tensor(label, dtype=torch.long),
        }

def collate_fn(batch):
    return {
        "input_values": torch.stack([b["input_values"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }

ds_train = SECDEnsembleDataset(df_train)
ds_val   = SECDEnsembleDataset(df_val)
ds_test  = SECDEnsembleDataset(df_test)

print(f"Train: {len(ds_train)}")
print(f"Val  : {len(ds_val)}")
print(f"Test : {len(ds_test)}")

# Quick sanity check
sample = ds_train[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"input_values shape: {sample['input_values'].shape}")
print(f"label: {sample['labels'].item()} ({FOCUS[sample['labels'].item()]})")

b = collate_fn([ds_train[0], ds_train[1]])
print(f"\nBatch input shape: {b['input_values'].shape}")
print(f"Batch labels shape: {b['labels'].shape}")

cell_end("Cell 11.11")

[Cell 11.11]  started  2026-04-11  10:05:46
Ensemble-size classes: ['duo', 'trio', 'quartet']
ens2i: {'duo': 0, 'trio': 1, 'quartet': 2}
Train: 183818
Val  : 39390
Test : 39390

Sample keys: ['input_values', 'labels']
input_values shape: torch.Size([128, 256])
label: 1 (trio)

Batch input shape: torch.Size([2, 128, 256])
Batch labels shape: torch.Size([2])
[Cell 11.11]  finished in 0.04 sec


## Cell 12 — Full Training Pipeline (EXP1: Ensemble-Size Classification)

This cell executes end-to-end model training for EXP1, including configuration, optimization, and checkpointing.
The best-performing checkpoint is retained for downstream validation and test evaluation.

In [ ]:
# Cell 12 — Full Training Pipeline (EXP1 / ensemble-size classification)

cell_start("Cell 12")

import os, time, json, warnings, socket, getpass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    PrinterCallback,
)
from transformers.utils import logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ── Self-contained helpers ──────────────────────────────────

def resize_ast_pos_embed(ast_model, target_frames: int):
    n_mels_patches = (128 - 16) // 10 + 1
    n_time_patches = (target_frames - 16) // 10 + 1
    n_patches = n_mels_patches * n_time_patches
    n_tokens = n_patches + 2
    pos_embed = ast_model.embeddings.position_embeddings
    old_n = pos_embed.shape[1]
    if old_n == n_tokens:
        return
    cls_dist = pos_embed[:, :2, :]
    patch_pos = pos_embed[:, 2:, :]
    old_freq, old_time, d = 12, 101, patch_pos.shape[-1]
    patch_pos = patch_pos.reshape(1, old_freq, old_time, d).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(
        patch_pos.float(),
        size=(n_mels_patches, n_time_patches),
        mode="bicubic",
        align_corners=False,
    )
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, n_patches, d)
    new_pos = torch.cat([cls_dist, patch_pos], dim=1)
    ast_model.embeddings.position_embeddings = nn.Parameter(new_pos)

def masked_ce(logits, targets, weight=None, ignore_index=-1):
    valid = targets.ne(ignore_index)
    if not valid.any():
        return logits.new_zeros(())
    return F.cross_entropy(
        logits[valid].float(),
        targets[valid],
        weight=weight.float() if weight is not None else None,
        label_smoothing=0.05,
    )

# ── Bind upstream objects ──

CLASS_WEIGHTS_T = CLASS_WEIGHTS.detach().clone().float().cpu()
train_dataset = ds_train
val_dataset   = ds_val
test_dataset  = ds_test
data_collator = collate_fn

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found.")

print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(SEED)
np.random.seed(SEED)

stamp = time.strftime("%Y%m%d-%H%M%S")
OUTDIR = os.path.join(paths.outdir, f"exp1_ast_ensemble_full_{stamp}")
os.makedirs(OUTDIR, exist_ok=True)

with open(os.path.join(paths.outdir, "latest_run.txt"), "w", encoding="utf-8") as f:
    f.write(OUTDIR)

with open(os.path.join(OUTDIR, "session_config.json"), "w", encoding="utf-8") as f:
    json.dump({
        "TASK_NAME": TASK_NAME,
        "TASK_OBJECTIVE": TASK_OBJECTIVE,
        "MODEL_NAME": MODEL_NAME,
        "FOCUS": list(FOCUS),
        "TARGET_FRAMES": int(TARGET_FRAMES),
        "CLASS_WEIGHTS": CLASS_WEIGHTS_T.tolist(),
    }, f, indent=2)

# ── Metrics ──

def compute_metrics_exp1(eval_pred):
    logits, labels = eval_pred
    logits = np.asarray(logits)
    labels = np.asarray(labels)

    preds = logits.argmax(axis=-1).reshape(-1)
    labels = labels.reshape(-1)

    mask = labels >= 0
    yt = labels[mask]
    yp = preds[mask]

    return {
        "eval_support": int(mask.sum()),
        "eval_acc": float(accuracy_score(yt, yp)),
        "eval_f1_macro": float(f1_score(yt, yp, average="macro", zero_division=0)),
    }

# ── Model ──

class ASTEnsembleSizeClassifier(nn.Module):
    def __init__(self, model_name, n_classes, class_weights):
        super().__init__()
        self.n_classes = n_classes

        self.ast = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        resize_ast_pos_embed(self.ast, TARGET_FRAMES)

        with torch.no_grad():
            dummy = torch.zeros(1, 128, TARGET_FRAMES)
            h = self.ast(input_values=dummy).last_hidden_state
            d = h.mean(dim=1).shape[-1]

        self.classifier = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(d, n_classes),
        )

        if isinstance(class_weights, torch.Tensor):
            cw = class_weights.detach().clone().float()
        else:
            cw = torch.tensor(class_weights, dtype=torch.float32)
        self.register_buffer("w_cls", cw)

    def feats(self, x):
        return self.ast(input_values=x).last_hidden_state.mean(dim=1)

    def forward(self, input_values, labels=None, **kwargs):
        emb = self.feats(input_values)
        logits = self.classifier(emb)

        loss = None
        if labels is not None:
            loss = masked_ce(logits, labels, weight=self.w_cls)

        return {"loss": loss, "logits": logits}

# ── Live progress callback ──

class EpochTableCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.train_losses.append(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return

        if state.epoch == 1:
            print("-" * 84)
            print("Epoch | Train Loss | Val Loss | Acc    | F1")
            print("-" * 84)

        ep = int(state.epoch or 0)
        tl = float(np.mean(self.train_losses)) if self.train_losses else 0.0
        self.train_losses = []

        vl  = float(metrics.get("eval_loss", 0.0))
        acc = float(metrics.get("eval_acc", 0.0))
        f1  = float(metrics.get("eval_f1_macro", 0.0))

        print(f"{ep:>5} | {tl:.4f}     | {vl:.4f}   | {acc:.4f} | {f1:.4f}")

# ── Build & launch ──

model = ASTEnsembleSizeClassifier(
    MODEL_NAME,
    len(FOCUS),
    CLASS_WEIGHTS_T,
).to(torch.device("cuda:0"))

model.ast.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=os.path.join(OUTDIR, "hf_ckpt"),
    logging_dir=os.path.join(OUTDIR, "logs"),

    num_train_epochs=40,
    per_device_train_batch_size=384,       # safe for L4; try 256–384 after checking VRAM
    per_device_eval_batch_size=512,

    learning_rate=3e-5,
    warmup_ratio=0.05,
    weight_decay=1e-2,
    bf16=True,                             # L4 supports bf16 — more stable than fp16
    dataloader_num_workers=8,              # utilise more vCPUs (try 12 if CPU-bound)
    dataloader_pin_memory=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    save_total_limit=2,

    remove_unused_columns=False,
    label_names=["labels"],
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_exp1,
    callbacks=[
        EpochTableCallback(),
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.001,
        ),
    ],
)

trainer.remove_callback(PrinterCallback)

print("-" * 72)
print(f"User : {getpass.getuser()} | Host : {socket.gethostname()}")
print(f"Run  : {OUTDIR}")
print("-" * 72)

t0 = time.time()
trainer.train()

print(f"\nTraining finished in {(time.time() - t0) / 60:.2f} min")

trainer.save_model(os.path.join(args.output_dir, "final"))
trainer.save_state()

print("-" * 72)
print("FULL RUN COMPLETE")
print(f"Artifacts -> {OUTDIR}")
print("-" * 72)

cell_end("Cell 12")


[Cell 12]  started  2026-04-11  10:14:36
GPU: NVIDIA L4


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 5295.91it/s]


------------------------------------------------------------------------
User : aggelosger | Host : audio-gpu-workstation
Run  : /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE/exp1_ast_ensemble_full_20260411-101436
------------------------------------------------------------------------
------------------------------------------------------------------------------------
Epoch | Train Loss | Val Loss | Acc    | F1
------------------------------------------------------------------------------------
    1 | 0.5976     | 0.3690   | 0.9021 | 0.9005
    2 | 0.3184     | 0.2659   | 0.9537 | 0.9529
    3 | 0.2625     | 0.2514   | 0.9622 | 0.9613
    4 | 0.2202     | 0.2231   | 0.9759 | 0.9755
    5 | 0.2039     | 0.2444   | 0.9669 | 0.9663
    6 | 0.1917     | 0.2424   | 0.9673 | 0.9663
    7 | 0.1886     | 0.2039   | 0.9856 | 0.9852
    8 | 0.1825     | 0.2047   | 0.9855 | 0.9852
    9 | 0.1804     | 0.2210   | 0.9789 | 0.9785
   10 | 0.1795     | 0.2042   | 0.9861 | 0.9857
   11 | 0.1774     | 0.20

## Cell 13 — Post-Training Evaluation and Artifacts (EXP1)

This cell loads the selected checkpoint and runs inference on validation and test sets.
It computes core metrics and exports the main artifacts required for reporting.

In [ ]:
# Cell 13 — Post-Training Evaluation and Artifacts (EXP1 ensemble-size task)

cell_start("Cell 13")

import os, json, glob, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from transformers import AutoModel, Trainer, TrainingArguments
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Self-contained helpers ──────────────────────────────────

def resize_ast_pos_embed(ast_model, target_frames: int):
    n_mels_patches = (128 - 16) // 10 + 1
    n_time_patches = (target_frames - 16) // 10 + 1
    n_patches = n_mels_patches * n_time_patches
    n_tokens = n_patches + 2
    pos_embed = ast_model.embeddings.position_embeddings
    old_n = pos_embed.shape[1]
    if old_n == n_tokens:
        return
    cls_dist = pos_embed[:, :2, :]
    patch_pos = pos_embed[:, 2:, :]
    old_freq, old_time, d = 12, 101, patch_pos.shape[-1]
    patch_pos = patch_pos.reshape(1, old_freq, old_time, d).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(
        patch_pos.float(),
        size=(n_mels_patches, n_time_patches),
        mode="bicubic",
        align_corners=False,
    )
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, n_patches, d)
    new_pos = torch.cat([cls_dist, patch_pos], dim=1)
    ast_model.embeddings.position_embeddings = nn.Parameter(new_pos)

def masked_ce(logits, targets, weight=None, ignore_index=-1):
    valid = targets.ne(ignore_index)
    if not valid.any():
        return logits.new_zeros(())
    return F.cross_entropy(
        logits[valid].float(),
        targets[valid],
        weight=weight.float() if weight is not None else None,
        label_smoothing=0.05,
    )

CLASS_WEIGHTS_T = CLASS_WEIGHTS.detach().clone().float().cpu()


if "OUTDIR" not in globals() or not os.path.isdir(globals().get("OUTDIR", "")):
    ptr = os.path.join(paths.outdir, "latest_run.txt")
    if not os.path.exists(ptr):
        raise RuntimeError("OUTDIR not found and latest_run.txt is missing.")
    with open(ptr, "r", encoding="utf-8") as f:
        OUTDIR = f.read().strip()
    print(f"Loaded run path -> {OUTDIR}")
else:
    print(f"Using OUTDIR -> {OUTDIR}")

CKPT_DIR = os.path.join(OUTDIR, "hf_ckpt")
ARTIFACT_DIR = os.path.join(OUTDIR, "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

cfg_path = os.path.join(OUTDIR, "session_config.json")
if not os.path.exists(cfg_path):
    raise RuntimeError("Missing session_config.json")

with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

FOCUS = list(cfg["FOCUS"])
TARGET_FRAMES = int(cfg["TARGET_FRAMES"])
MODEL_NAME = cfg["MODEL_NAME"]
W_CLS = np.array(cfg["CLASS_WEIGHTS"], dtype=np.float32)
N_CLASSES = len(FOCUS)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device -> {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

need = ["val_dataset", "test_dataset", "data_collator"]
for req in need:
    if req not in globals():
        raise RuntimeError(f"Missing: {req}")

print(f"VAL: {len(val_dataset):,} | TEST: {len(test_dataset):,}")

class ASTEnsembleSizeClassifier(nn.Module):
    def __init__(self, model_name, n_classes, class_weights):
        super().__init__()
        self.n_classes = n_classes

        self.ast = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        resize_ast_pos_embed(self.ast, TARGET_FRAMES)

        with torch.no_grad():
            dummy = torch.zeros(1, 128, TARGET_FRAMES)
            h = self.ast(input_values=dummy).last_hidden_state
            d = h.mean(dim=1).shape[-1]

        self.classifier = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(d, n_classes),
        )
        self.register_buffer("w_cls", torch.tensor(class_weights, dtype=torch.float32))

    def feats(self, x):
        return self.ast(input_values=x).last_hidden_state.mean(dim=1)

    def forward(self, input_values, labels=None, **kwargs):
        emb = self.feats(input_values)
        logits = self.classifier(emb)

        loss = None
        if labels is not None:
            loss = masked_ce(logits, labels, weight=self.w_cls)
        return {"loss": loss, "logits": logits}

def resolve_best_checkpoint(ckpt_dir):
    state_path = os.path.join(ckpt_dir, "trainer_state.json")
    if not os.path.exists(state_path):
        return None
    with open(state_path, "r", encoding="utf-8") as f:
        state = json.load(f)
    best = state.get("best_model_checkpoint")
    if best and os.path.isdir(best):
        print(f"BEST checkpoint -> {best}")
        return best
    return None

model_dir = resolve_best_checkpoint(CKPT_DIR)

if model_dir is None:
    checkpoints = sorted(
        glob.glob(os.path.join(CKPT_DIR, "checkpoint-*")),
        key=lambda x: int(x.split("-")[-1])
    )
    if not checkpoints:
        raise RuntimeError("No checkpoints found")
    model_dir = checkpoints[-1]
    print(f"Fallback -> {model_dir}")

import safetensors.torch as st

model = ASTEnsembleSizeClassifier(MODEL_NAME, N_CLASSES, W_CLS)

state_dict = st.load_file(
    os.path.join(model_dir, "model.safetensors"),
    device=str(device)
)
load_info = model.load_state_dict(state_dict, strict=False)
if load_info.missing_keys:
    print(f"Missing keys    : {load_info.missing_keys}")
if load_info.unexpected_keys:
    print(f"Unexpected keys : {load_info.unexpected_keys}")

model.to(device)
model.eval()

eval_trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=ARTIFACT_DIR,
        per_device_eval_batch_size=64,
        report_to=[],
        remove_unused_columns=False,
        label_names=["labels"],
        use_cpu=(device.type == "cpu"),
        seed=SEED,
    ),
    data_collator=data_collator,
)

def run_predictions(dataset):
    pred = eval_trainer.predict(dataset)
    raw_logits = pred.predictions
    raw_labels = pred.label_ids

    logits = np.asarray(raw_logits[0]) if isinstance(raw_logits, tuple) else np.asarray(raw_logits)
    labels = np.asarray(raw_labels[0]) if isinstance(raw_labels, tuple) else np.asarray(raw_labels)

    pred_ids = logits.argmax(axis=-1).reshape(-1)
    labels = labels.reshape(-1)

    mask = labels >= 0
    y_true = labels[mask]
    y_pred = pred_ids[mask]

    return y_true, y_pred, logits

def save_cm(y_true, y_pred, labels, title, path, normalize=None):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(labels))),
        normalize=normalize
    )
    fig, ax = plt.subplots(figsize=(7, 6))
    fmt = ".2f" if normalize is not None else "d"
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
        values_format=fmt
    )
    ax.set_title(title, fontsize=12, pad=10)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

all_metrics = {}

for split_name, dataset in [("val", val_dataset), ("test", test_dataset)]:
    print(f"\n=== {split_name.upper()} ===")

    y_true, y_pred, logits = run_predictions(dataset)

    rep = classification_report(
        y_true,
        y_pred,
        target_names=FOCUS,
        output_dict=True,
        zero_division=0,
    )

    all_metrics[split_name] = rep

    print(f"{split_name} | acc={rep['accuracy']:.4f} | f1={rep['macro avg']['f1-score']:.4f}")

    save_cm(
        y_true, y_pred, FOCUS,
        f"{split_name.upper()} | Ensemble size — Absolute",
        os.path.join(ARTIFACT_DIR, f"cm_{split_name}_abs.png"),
        normalize=None,
    )
    save_cm(
        y_true, y_pred, FOCUS,
        f"{split_name.upper()} | Ensemble size — Normalized",
        os.path.join(ARTIFACT_DIR, f"cm_{split_name}_norm.png"),
        normalize="true",
    )

with open(os.path.join(ARTIFACT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2)

pngs = sorted(f for f in os.listdir(ARTIFACT_DIR) if f.endswith(".png"))
print(f"\nArtifacts saved -> {ARTIFACT_DIR}")
print(f"metrics.json + {len(pngs)} PNGs")
for p in pngs:
    print(f"  {p}")

cell_end("Cell 13")


[Cell 13]  started  2026-04-11  15:54:48
Using OUTDIR -> /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE/exp1_ast_ensemble_full_20260411-101436
Device -> cuda:0 (NVIDIA L4)
VAL: 39,390 | TEST: 39,390
BEST checkpoint -> /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE/exp1_ast_ensemble_full_20260411-101436/hf_ckpt/checkpoint-5748


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 5602.80it/s]



=== VAL ===
val | acc=0.9865 | f1=0.9862

=== TEST ===
test | acc=0.9867 | f1=0.9864

Artifacts saved -> /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE/exp1_ast_ensemble_full_20260411-101436/artifacts
metrics.json + 4 PNGs
  cm_test_abs.png
  cm_test_norm.png
  cm_val_abs.png
  cm_val_norm.png
[Cell 13]  finished in 777.79 sec


## Cell 13.05 — Training Curves (Loss and Macro-F1)

This cell reads training history and produces learning curves for loss and macro-F1.
The plots are used to summarize optimization behavior and convergence quality.

In [ ]:
# Cell 13.05 — Training Curves (Loss + F1)

cell_start("Cell 13.05")

import os, json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ARTIFACT_DIR = os.path.join(OUTDIR, "artifacts")
CKPT_DIR = os.path.join(OUTDIR, "hf_ckpt")

state_path = os.path.join(CKPT_DIR, "trainer_state.json")
if not os.path.exists(state_path):
    raise RuntimeError(f"trainer_state.json not found in {CKPT_DIR}")

with open(state_path, "r", encoding="utf-8") as f:
    state = json.load(f)

log = state["log_history"]

train_epochs, train_loss = [], []
val_epochs, val_loss, val_acc, val_f1 = [], [], [], []

for entry in log:
    e = entry.get("epoch")
    if e is None:
        continue
    if "loss" in entry and "eval_loss" not in entry:
        train_epochs.append(e)
        train_loss.append(entry["loss"])
    if "eval_loss" in entry:
        val_epochs.append(e)
        val_loss.append(entry["eval_loss"])
        val_acc.append(entry.get("eval_acc", None))
        val_f1.append(entry.get("eval_f1_macro", None))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_epochs, train_loss, "o-", ms=3, label="Train loss")
ax.plot(val_epochs, val_loss, "s-", ms=3, label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, "loss_curves.png"), dpi=150)
plt.close(fig)
print("loss_curves.png")

has_acc = any(v is not None for v in val_acc)
has_f1  = any(v is not None for v in val_f1)

if has_acc or has_f1:
    fig, ax = plt.subplots(figsize=(9, 5))
    if has_acc:
        ax.plot(val_epochs, val_acc, "s-", ms=3, label="Val accuracy")
    if has_f1:
        ax.plot(val_epochs, val_f1, "^-", ms=3, label="Val macro F1")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    ax.set_title("Validation Accuracy & Macro F1")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(ARTIFACT_DIR, "f1_curves.png"), dpi=150)
    plt.close(fig)
    print("f1_curves.png")
else:
    print("No eval_acc / eval_f1_macro found — f1_curves.png skipped")

print(f"Saved -> {ARTIFACT_DIR}")

cell_end("Cell 13.05")


[Cell 13.05]  started  2026-04-11  16:07:46
loss_curves.png
f1_curves.png
Saved -> /home/aggelosger/EXP1_AST_ENSEMBLE_SIZE/exp1_ast_ensemble_full_20260411-101436/artifacts
[Cell 13.05]  finished in 0.34 sec


## Cell 13.08 — Classification Reports from `metrics.json`

This cell loads saved evaluation outputs and presents class-wise precision, recall, and F1 statistics.
It complements the aggregate metrics reported in the experiment summary.

In [ ]:
#Cell 13.08 Load and Print Classification Reports from metrics.json

import json
from pathlib import Path

# Βρες το metrics.json
json_path = os.path.join(ARTIFACT_DIR, "metrics.json")

if not os.path.exists(json_path):
    print("❌ metrics.json not found!")
else:
    with open(json_path, "r", encoding="utf-8") as f:
        all_metrics = json.load(f)

    print("="*65)
    print("CLASSIFICATION REPORTS FROM SAVED METRICS.JSON")
    print("="*65)

    for split in ["val", "test"]:
        if split not in all_metrics:
            continue

        data = all_metrics[split]
        print(f"\n\n=== {split.upper()} SET CLASSIFICATION REPORT ===\n")

        # Pretty table
        print(f"{'Class':<12} {'Precision':>9} {'Recall':>9} {'F1-Score':>9} {'Support':>8}")
        print("-" * 65)

        for cls in FOCUS:
            m = data[cls]
            print(f"{cls:<12} {m['precision']:9.4f} {m['recall']:9.4f} {m['f1-score']:9.4f} {int(m['support']):8d}")

        # Macro / Weighted / Accuracy
        macro = data['macro avg']
        acc = data['accuracy']

        print("-" * 65)
        print(f"{'Accuracy':<12} {acc:9.4f}")
        print(f"{'Macro Avg':<12} {macro['precision']:9.4f} {macro['recall']:9.4f} {macro['f1-score']:9.4f}")
        print("="*65)


CLASSIFICATION REPORTS FROM SAVED METRICS.JSON


=== VAL SET CLASSIFICATION REPORT ===

Class        Precision    Recall  F1-Score  Support
-----------------------------------------------------------------
duo             0.9927    0.9933    0.9930    13500
trio            0.9823    0.9753    0.9788    12000
quartet         0.9841    0.9896    0.9869    13890
-----------------------------------------------------------------
Accuracy        0.9865
Macro Avg       0.9864    0.9861    0.9862


=== TEST SET CLASSIFICATION REPORT ===

Class        Precision    Recall  F1-Score  Support
-----------------------------------------------------------------
duo             0.9922    0.9936    0.9929    13500
trio            0.9812    0.9768    0.9790    12000
quartet         0.9861    0.9886    0.9873    13890
-----------------------------------------------------------------
Accuracy        0.9867
Macro Avg       0.9865    0.9863    0.9864
